# Flow Matching

**Flow matching** is the deterministic transport counterpart of the diffusion viewpoint. A stochastic noising process can often be viewed through an associated **probability flow ODE**. Flow matching starts directly from that deterministic viewpoint. Instead of first introducing randomness and then deriving a transport equation, it chooses a path between a simple source distribution and the data distribution, then learns the **velocity field** that realizes that path.

This is why flow matching feels like a natural continuation of several earlier ideas. VAEs emphasize hidden structure and low-dimensional organization. GANs emphasize generators as distribution-shaping mechanisms rather than fixed formulas. Diffusion emphasizes **time-dependent paths** and local prediction targets. Flow matching keeps the path-based view, removes the stochastic wrapper, and turns learning into a clean regression problem on transport velocities.


```{figure} ../assets/images/FM.png
:width: 78%
:align: center

A conceptual view of flow matching: start from a simple source distribution, prescribe a family of intermediate distributions, and learn the deterministic velocity field that pushes mass along that path until it reaches the data distribution.
```


The flow of the discussion mirrors the structure used elsewhere: first the deterministic-transport language, then the conditional-path construction, then a guided implementation. The first implementation is intentionally two-dimensional because transport is easiest to understand when the moving point cloud can be plotted directly. After that, the same ideas are scaled to **FashionMNIST** with a small image-space rectified-flow model.


```{important}
Flow matching separates two questions that are often entangled elsewhere: **which probability path should connect source and data**, and **which velocity field realizes that path**. The first is a modeling choice. The second is the learning problem.
```


## Deterministic Transport and the Continuity Equation

Let $\boldsymbol{x}(t)$ solve the ODE $\frac{d\boldsymbol{x}}{dt} = \boldsymbol{v}(\boldsymbol{x}(t), t)$, where $\boldsymbol{v}$ is a time-dependent velocity field. If the initial condition $\boldsymbol{x}(0)$ is random with density $p_0$, then the density $p_t$ of $\boldsymbol{x}(t)$ changes over time according to a conservation law:
:::{math}
\partial_t p_t(\boldsymbol{x}) + \nabla_{\boldsymbol{x}} \cdot \big(p_t(\boldsymbol{x})\,\boldsymbol{v}(\boldsymbol{x}, t)\big) = 0.
:::

This is the **continuity equation**. It says that probability mass is not created or destroyed; it is simply redistributed by the flow.


```{prf:theorem} Continuity equation for deterministic flows
:label: thm-flow-continuity

Suppose $\boldsymbol{x}(t)$ solves
:::{math}
\frac{d\boldsymbol{x}}{dt} = \boldsymbol{v}(\boldsymbol{x}(t), t)
:::
and let $p_t$ denote the density of $\boldsymbol{x}(t)$. Under standard regularity assumptions,
:::{math}
\partial_t p_t(\boldsymbol{x}) + \nabla_{\boldsymbol{x}} \cdot \big(p_t(\boldsymbol{x})\,\boldsymbol{v}(\boldsymbol{x}, t)\big) = 0.
:::
```

```{prf:proof}
Let $\varphi$ be a smooth compactly supported test function. By the chain rule,
:::{math}
\frac{d}{dt}\mathbb{E}[\varphi(\boldsymbol{x}(t))]
=
\mathbb{E}\left[\nabla \varphi(\boldsymbol{x}(t))^\top \boldsymbol{v}(\boldsymbol{x}(t), t)\right].
:::
Writing the expectation against the density $p_t$ gives
:::{math}
\frac{d}{dt}\int \varphi(\boldsymbol{x}) p_t(\boldsymbol{x}) \, d\boldsymbol{x}
=
\int \nabla \varphi(\boldsymbol{x})^\top \boldsymbol{v}(\boldsymbol{x}, t) p_t(\boldsymbol{x}) \, d\boldsymbol{x}.
:::
Integrating by parts moves the gradient onto the flux term and yields
:::{math}
\int \varphi(\boldsymbol{x})
\left[
    \partial_t p_t(\boldsymbol{x}) + \nabla_{\boldsymbol{x}} \cdot \big(p_t(\boldsymbol{x})\,\boldsymbol{v}(\boldsymbol{x}, t)\big)
\right] d\boldsymbol{x} = 0.
:::
Since this holds for every test function $\varphi$, the bracketed quantity must vanish.
```


The physical picture is often clearer than the PDE. Imagine a colored dye moved by a fluid. The dye concentration changes at a location because the fluid carries mass away or brings mass in, not because matter appears from nowhere. Probability density behaves in the same way. Flow matching therefore asks a very geometric question: **what velocity field should move mass from the simple source distribution to the data distribution?**


## Probability Paths and Why They Matter

The next ingredient is a **probability path** $(p_t)_{t \in [0,1]}$ with $p_0$ easy to sample and $p_1 = p_{gt}$. In practice, $p_0$ is often a standard Gaussian and $p_{gt}$ is the data distribution. Once the path is chosen, the modeling problem becomes: learn a vector field whose induced flow pushes $p_0$ through those intermediate marginals and lands on $p_{gt}$ at time one.

The important conceptual separation is this: in flow matching, **path design** and **field learning** are different decisions. That is one reason the framework is elegant. The path is a modeling choice. The vector field is the trainable object.


A direct way to phrase the method is the following. Diffusion says: “a corruption process is known, and the reverse of that corruption is learned.” Flow matching says: “a connection between source and target examples is defined directly, and the velocity consistent with that connection is learned.” Both are path-based generative models, but the object being predicted is different. Diffusion predicts noise or score-related quantities. Flow matching predicts **transport velocities**.


## Conditional Probability Paths

Let $\boldsymbol{x}_0 \sim p_0$ and $\boldsymbol{x}_1 \sim p_{gt}$. A simple conditional path is the linear interpolation
:::{math}
\boldsymbol{x}_t = (1-t)\boldsymbol{x}_0 + t\boldsymbol{x}_1.
:::
Conditionally on the endpoints, its instantaneous velocity is
:::{math}
\frac{d\boldsymbol{x}_t}{dt} = \boldsymbol{x}_1 - \boldsymbol{x}_0.
:::
This is already enough to generate a useful supervised target for learning.


```{admonition} Numerical Example: A Straight Conditional Path
:class: numerical-example

Let $\boldsymbol{x}_0 = (0,0)$ and $\boldsymbol{x}_1 = (4,2)$. Then the linear path is $\boldsymbol{x}_t = (4t, 2t)$. At time $t = 0.25$, the intermediate point is $(1, 0.5)$ and the conditional velocity is still $(4,2)$. In other words, the local instruction given to the model is: “if you are at $(1,0.5)$ at time $0.25$ for this endpoint pair, keep moving in the direction $(4,2)$.”

One such instruction is trivial. The power of flow matching comes from averaging many such local instructions across many endpoint pairs until a coherent global transport field emerges.
```


This is the first point where it becomes clear that flow matching is **not** just naive interpolation. A single pair of endpoints defines a single path. The learned vector field, however, must be globally consistent across many different endpoint pairs that all pass through overlapping regions of space at different times. The model is learning a transport law, not memorizing one line per sample.


```{prf:theorem} Conditional flow matching target
:label: thm-cfm-regression

Let $(\boldsymbol{x}_0, \boldsymbol{x}_1)$ be sampled from a coupling of $p_0$ and $p_{gt}$. Let $\boldsymbol{x}_t$ be sampled from a conditional interpolation law $p_t(\boldsymbol{x} | \boldsymbol{x}_0, \boldsymbol{x}_1)$ with conditional velocity target $\boldsymbol{u}_t(\boldsymbol{x} | \boldsymbol{x}_0, \boldsymbol{x}_1)$. Then the minimizer of
:::{math}
\mathcal{L}_{CFM}(\theta)
=
\mathbb{E}\left[
    \|\boldsymbol{v}_\theta(\boldsymbol{x}_t, t) - \boldsymbol{u}_t(\boldsymbol{x}_t | \boldsymbol{x}_0, \boldsymbol{x}_1)\|_2^2
\right]
:::
is
:::{math}
\boldsymbol{v}^\star(\boldsymbol{x}, t)
=
\mathbb{E}\big[\boldsymbol{u}_t(\boldsymbol{x}_t | \boldsymbol{x}_0, \boldsymbol{x}_1) \mid \boldsymbol{x}_t = \boldsymbol{x}\big].
:::
```

```{prf:proof}
For fixed $(\boldsymbol{x}, t)$, the loss is a squared-error regression problem whose random target is the conditional velocity. The minimizer of squared error is the conditional expectation of that target given the input. Applying that fact pointwise yields the claim.
```


This theorem is short, but it contains the practical heart of the method. Once the conditional path is chosen, the generative-learning problem becomes a regression problem. That is the same structural miracle we saw in diffusion: a difficult distribution-matching objective becomes an ordinary supervised target. The difference is that the target is now a **velocity field** rather than a denoising target.


## Relation with Continuous Normalizing Flows and Diffusion

Continuous normalizing flows also use an ODE to transform a source distribution into a target distribution, but they are usually trained by maximum likelihood. That means computing divergence terms and integrating dynamics inside the training objective itself. Flow matching keeps the ODE-based sampling viewpoint but drops the expensive likelihood machinery during training. It says: choose a path, compute the corresponding local transport target, and regress a neural field toward that target.

From the diffusion side, the connection is just as close. Diffusion begins with a stochastic path and can often be interpreted through a related deterministic **probability flow ODE**. Flow matching starts from the deterministic path directly. This is why the two families feel so closely related in practice. Both learn time-dependent neural fields that move mass from a simple source to the data distribution. They differ mostly in how the target field is constructed.


```{note}
The regression target is not “the velocity for one specific endpoint pair forever.” The learned field is the **conditional expectation** of many local transport targets given the current state and time. That is why one global vector field can emerge from many sampled endpoint pairs.
```


## Path Design: Linear, Gaussian, and Transport-Inspired Choices

The straight line path is the simplest choice, but it is not the only one. One can use **Gaussian paths** whose mean interpolates between source and target while the variance changes with time, or paths inspired by **optimal transport** so that the conditional trajectories are straighter and easier to learn. This design choice matters because it changes the local regression targets that the model sees during training.

A generative model is therefore not only a neural architecture. It is also a choice of geometry. In VAEs that geometry lives in the latent variable and the KL regularizer. In diffusion it lives in the noising process. In flow matching it lives directly in the chosen probability path.


## Guided Implementation: Seeing Transport in Two Dimensions

The first implementation uses a deliberately visible example. The target distribution is a noisy two-ring dataset, the source is a standard Gaussian, and the model learns a time-dependent velocity field that transports one into the other. The example is small, but it makes it possible to inspect both the final point cloud and the trajectories used to get there.


In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, utils
from tqdm.auto import tqdm
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(7)
if device.type == "cuda":
    torch.cuda.manual_seed_all(7)


The first design choice is the pair of endpoint distributions. We want the source to be easy to sample and the target to be structured enough that the transport is visually interesting. Two noisy concentric rings work well because a good model must learn a genuinely multimodal mapping instead of merely shifting one cloud by a constant offset.


In [ ]:
batch_size = 512
hidden_dim = 192
time_dim = 96
toy_steps = 4000
toy_lr = 8e-4


def sample_target(batch_size, device):
    angles = 2 * math.pi * torch.rand(batch_size, device=device)
    radii = torch.where(
        torch.rand(batch_size, device=device) > 0.5,
        torch.full((batch_size,), 2.0, device=device),
        torch.full((batch_size,), 4.0, device=device),
    )
    noise = 0.15 * torch.randn(batch_size, 2, device=device)
    points = torch.stack([radii * torch.cos(angles), radii * torch.sin(angles)], dim=1)
    return points + noise


def sample_source(batch_size, device):
    return torch.randn(batch_size, 2, device=device)


def sample_linear_path(batch_size, device):
    x0 = sample_source(batch_size, device)
    x1 = sample_target(batch_size, device)
    t = torch.rand(batch_size, 1, device=device)
    # Straight interpolation makes the training target analytically explicit.
    xt = (1.0 - t) * x0 + t * x1
    ut = x1 - x0
    return x0, x1, xt, ut, t


The code above is the practical incarnation of the theory. The conditional path defines both the intermediate state $\boldsymbol{x}_t$ and the target velocity. In other words, the entire learning signal comes from the path design. There is no discriminator, no variational bound, and no explicit likelihood term hidden in the objective.


In [ ]:
class TimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half_dim = self.dim // 2
        factor = math.log(10000.0) / max(half_dim - 1, 1)
        frequencies = torch.exp(torch.arange(half_dim, device=t.device) * -factor)
        angles = t * frequencies.unsqueeze(0)
        emb = torch.cat([angles.sin(), angles.cos()], dim=1)
        if self.dim % 2 == 1:
            emb = F.pad(emb, (0, 1))
        return emb


class VelocityField(nn.Module):
    def __init__(self, hidden_dim=192, time_dim=96):
        super().__init__()
        self.time_embedding = nn.Sequential(
            TimeEmbedding(time_dim),
            nn.Linear(time_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.net = nn.Sequential(
            nn.Linear(hidden_dim + 2, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, x, t):
        t_emb = self.time_embedding(t)
        return self.net(torch.cat([x, t_emb], dim=1))


toy_model = VelocityField(hidden_dim=hidden_dim, time_dim=time_dim).to(device)
toy_optimizer = torch.optim.Adam(toy_model.parameters(), lr=toy_lr)


def toy_flow_matching_loss(model, batch_size, device):
    _, _, xt, ut, t = sample_linear_path(batch_size, device)
    pred = model(xt, t)
    return F.mse_loss(pred, ut)


Time conditioning plays a role here that is close to diffusion but not identical. In diffusion, time mostly indexes **how corrupted** the sample is. In flow matching, time indexes **where along the transport path** the sample currently lies. The network therefore needs to learn not only where mass should move, but also how that movement changes between early and late stages of the path.


In [ ]:
toy_history = []

for step in tqdm(range(toy_steps), desc="Toy flow-matching steps"):
    toy_optimizer.zero_grad()
    loss = toy_flow_matching_loss(toy_model, batch_size, device)
    loss.backward()
    toy_optimizer.step()

    toy_history.append(loss.item())
    if (step + 1) % 200 == 0:
        tqdm.write(f"Step {step + 1:04d} | loss: {loss.item():.6f}")


Even in this tiny example, it is useful to distinguish **fitting the field** from **solving the field**. Training only tells us whether the network is matching the local velocity targets sampled from the path construction. Sampling quality will also depend on the numerical ODE solver. A good field with a poor solver can still yield mediocre generations.


In [ ]:
@torch.no_grad()
def sample_toy_trajectory(model, n_samples=2000, steps=100):
    model.eval()
    x = sample_source(n_samples, device)
    trajectory = [x.detach().cpu()]
    dt = 1.0 / steps

    for i in tqdm(range(steps), desc="Toy ODE sampling", leave=False):
        t = torch.full((n_samples, 1), i / steps, device=device)
        v = model(x, t)
        # Euler integration keeps the solver especially easy to inspect.
        x = x + dt * v
        trajectory.append(x.detach().cpu())

    return trajectory


toy_trajectory = sample_toy_trajectory(toy_model, n_samples=2000, steps=100)
toy_source = toy_trajectory[0]
toy_generated = toy_trajectory[-1]
toy_target = sample_target(2000, device).cpu()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].scatter(toy_source[:, 0], toy_source[:, 1], s=4, alpha=0.5)
axes[0].set_title("Source")
axes[0].axis("equal")

axes[1].scatter(toy_generated[:, 0], toy_generated[:, 1], s=4, alpha=0.5)
axes[1].set_title("Generated")
axes[1].axis("equal")

axes[2].scatter(toy_target[:, 0], toy_target[:, 1], s=4, alpha=0.5)
axes[2].set_title("Target")
axes[2].axis("equal")

plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(toy_history)
plt.xlabel("Training step")
plt.ylabel("Flow-matching loss")
plt.tight_layout()
plt.show()


When this section works, the point cloud should visibly migrate from a Gaussian source toward the two-ring target, and the loss should decrease steadily. If the model reaches only one ring, the field may be undertrained or too narrow. If the trajectories look jagged, the issue may be solver accuracy rather than field quality. These are the same kinds of distinctions that matter later for image-scale models: transport quality is a combination of **representation**, **training**, and **numerical integration**.


## From Toy Transport to Image Generation

The two-dimensional example clarifies the geometry, but the image-scale version of flow matching is usually presented as **rectified flow** or a close variant. A noise tensor $\boldsymbol{x}_0$ with the same shape as an image is paired with a data image $\boldsymbol{x}_1$, interpolated in time, and passed to a time-conditioned image model that predicts the transport velocity $\boldsymbol{x}_1 - \boldsymbol{x}_0$.

At a high level, the implementation starts to look surprisingly close to diffusion: image tensors, time embeddings, a U-Net-like architecture, random times, and an MSE training target. The difference is that diffusion predicts denoising-related quantities associated with a stochastic path, whereas rectified flow predicts a **deterministic velocity field**.


## Rectified Flow on FashionMNIST

FashionMNIST is a convenient bridge dataset because it is small enough to train quickly while still rich enough to produce recognizable structure. The images are scaled to $[-1,1]$ so that the data are centered like the Gaussian source. That makes the interpolation path numerically better behaved and keeps the network from having to learn a large mean shift before it learns meaningful transport.


In [ ]:
image_batch_size = 128
image_size = 28
image_channels = 1
image_base_channels = 64
image_time_dim = 128
image_epochs = 35
image_lr = 2e-4
image_ode_steps = 100
num_workers = 0

project_root = Path.cwd() if (Path.cwd() / "_config.yml").exists() else Path.cwd().parent
DATA_ROOT = project_root / "data"

image_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: 2.0 * x - 1.0),
])

fashion_train = datasets.FashionMNIST(
    root=DATA_ROOT,
    train=True,
    download=True,
    transform=image_transform,
)

fashion_loader = DataLoader(
    fashion_train,
    batch_size=image_batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=(device.type == "cuda"),
)


The architecture below is deliberately close to the compact U-Net family already used in diffusion. That is not an accident. Once the input is an image tensor and the target depends on time, the architectural needs are similar: local spatial processing, multiscale context, and a way to inject time information at several depths.


In [ ]:
class ImageResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.time_mlp = nn.Linear(time_dim, out_channels)
        self.norm1 = nn.GroupNorm(8, out_channels)
        self.norm2 = nn.GroupNorm(8, out_channels)
        self.activation = nn.SiLU()
        self.residual = (
            nn.Conv2d(in_channels, out_channels, kernel_size=1)
            if in_channels != out_channels else nn.Identity()
        )

    def forward(self, x, t_emb):
        h = self.conv1(x)
        h = self.norm1(h)
        h = self.activation(h)
        h = h + self.time_mlp(t_emb).unsqueeze(-1).unsqueeze(-1)
        h = self.conv2(h)
        h = self.norm2(h)
        h = self.activation(h)
        return h + self.residual(x)


class ImageVelocityUNet(nn.Module):
    def __init__(self, in_channels=1, base_channels=64, time_dim=128):
        super().__init__()
        self.time_embedding = nn.Sequential(
            TimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )
        self.input_conv = nn.Conv2d(in_channels, base_channels, kernel_size=3, padding=1)
        self.down1 = ImageResidualBlock(base_channels, base_channels, time_dim)
        self.downsample1 = nn.Conv2d(base_channels, base_channels * 2, kernel_size=4, stride=2, padding=1)
        self.down2 = ImageResidualBlock(base_channels * 2, base_channels * 2, time_dim)
        self.downsample2 = nn.Conv2d(base_channels * 2, base_channels * 4, kernel_size=4, stride=2, padding=1)
        self.down3 = ImageResidualBlock(base_channels * 4, base_channels * 4, time_dim)
        self.mid = ImageResidualBlock(base_channels * 4, base_channels * 4, time_dim)
        self.upsample2 = nn.ConvTranspose2d(base_channels * 4, base_channels * 2, kernel_size=4, stride=2, padding=1)
        self.up2 = ImageResidualBlock(base_channels * 4, base_channels * 2, time_dim)
        self.upsample1 = nn.ConvTranspose2d(base_channels * 2, base_channels, kernel_size=4, stride=2, padding=1)
        self.up1 = ImageResidualBlock(base_channels * 2, base_channels, time_dim)
        self.output_conv = nn.Conv2d(base_channels, in_channels, kernel_size=1)

    def forward(self, x, t):
        t_emb = self.time_embedding(t)
        x0 = self.input_conv(x)
        x1 = self.down1(x0, t_emb)
        x2 = self.downsample1(x1)
        x2 = self.down2(x2, t_emb)
        x3 = self.downsample2(x2)
        x3 = self.down3(x3, t_emb)
        x_mid = self.mid(x3, t_emb)
        x_up = self.upsample2(x_mid)
        x_up = torch.cat([x_up, x2], dim=1)
        x_up = self.up2(x_up, t_emb)
        x_up = self.upsample1(x_up)
        x_up = torch.cat([x_up, x1], dim=1)
        x_up = self.up1(x_up, t_emb)
        return self.output_conv(x_up)


image_flow_model = ImageVelocityUNet(
    in_channels=image_channels,
    base_channels=image_base_channels,
    time_dim=image_time_dim,
).to(device)
image_optimizer = torch.optim.AdamW(image_flow_model.parameters(), lr=image_lr, weight_decay=1e-4)


A useful way to read this architecture is to compare it mentally with the diffusion U-Net. The convolutional backbone is similar because the input is still an image and the network still needs spatial hierarchy. The *meaning* of the output, however, is different. The final layer is not predicting Gaussian noise or a decoder mean. It is predicting the **velocity vector** telling us how the current tensor should move through image space.


In [ ]:
def sample_image_path(x1):
    x0 = torch.randn_like(x1)
    t = torch.rand(x1.size(0), 1, 1, 1, device=x1.device)
    xt = (1.0 - t) * x0 + t * x1
    ut = x1 - x0
    return x0, x1, xt, ut, t.view(x1.size(0), 1)


def rectified_flow_loss(model, x1):
    _, _, xt, ut, t = sample_image_path(x1)
    pred = model(xt, t)
    return F.mse_loss(pred, ut)


This loss is one of the nicest parts of the method. Once the conditional path has been chosen, the image-scale objective is still just mean squared error. That does **not** mean the model is simplistic. It means the modeling sophistication has been moved into the path design, the architecture, and the ODE-based sampling stage rather than into a more baroque objective.


In [ ]:
image_history = []

for epoch in tqdm(range(image_epochs), desc="Image rectified-flow epochs"):
    image_flow_model.train()
    running_loss = 0.0

    for x1, _ in tqdm(fashion_loader, desc="train", leave=False):
        x1 = x1.to(device)
        image_optimizer.zero_grad()
        loss = rectified_flow_loss(image_flow_model, x1)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(image_flow_model.parameters(), max_norm=1.0)
        image_optimizer.step()
        running_loss += loss.item()

    epoch_loss = running_loss / len(fashion_loader)
    image_history.append(epoch_loss)
    print(f"Epoch {epoch + 1:02d} | rectified-flow loss: {epoch_loss:.6f}")


The loss curve is interpretable in the same way as the diffusion MSE: lower is usually better, but the scalar is not the whole story. A model can fit the local regression targets reasonably well and still generate weak samples if the sampler is too coarse or if the learned field is less accurate along its own generated trajectories than on training interpolants.


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(image_history)
plt.xlabel("Epoch")
plt.ylabel("Rectified-flow loss")
plt.tight_layout()
plt.show()


@torch.no_grad()
def sample_rectified_flow(model, n_samples=16, steps=image_ode_steps, show_progress=True):
    model.eval()
    x = torch.randn(n_samples, image_channels, image_size, image_size, device=device)
    dt = 1.0 / steps
    iterator = range(steps)
    if show_progress:
        iterator = tqdm(iterator, desc="rectified-flow sampling", leave=False)

    for i in iterator:
        t = torch.full((n_samples, 1), i / steps, device=device)
        v = model(x, t)
        # A midpoint step is a simple but meaningful upgrade over raw Euler.
        x_mid = x + 0.5 * dt * v
        t_mid = torch.full((n_samples, 1), (i + 0.5) / steps, device=device)
        x = x + dt * model(x_mid, t_mid)

    x = x.clamp(-1.0, 1.0)
    return 0.5 * (x + 1.0).cpu()


image_samples = sample_rectified_flow(image_flow_model, n_samples=16)
image_grid = utils.make_grid(image_samples, nrow=4, pad_value=1.0)
plt.figure(figsize=(6, 6))
plt.imshow(image_grid.permute(1, 2, 0), cmap="gray")
plt.axis("off")
plt.show()


Midpoint integration is already a hint of the engineering story that matters for continuous-time generators. Once the field has been trained, sample quality depends on the ODE solver as well as on the network. This is one of the clearest examples of the distinction between **learning the local rule** and **numerically realizing the global trajectory**.


## Evaluating Samples with FID and KID

At image scale, the evaluation logic is the same as for VAEs, GANs, and diffusion. We compare real and generated samples in a pretrained feature space. The main practical caution is batching: both real and fake images should be streamed through the metric objects in moderate minibatches rather than materialized all at once. We also keep the real features cached so repeated evaluations do not redo unnecessary work.


```{note}
At image scale, flow matching starts to look architecturally similar to diffusion: image tensors, time embeddings, U-Net-like processing, and regression losses. The real difference is **what the network predicts**: a transport velocity rather than noise or a score.
```


In [ ]:
def prepare_for_inception_metrics(images):
    if images.size(1) == 1:
        images = images.repeat(1, 3, 1, 1)
    return images.clamp(0.0, 1.0)


@torch.no_grad()
def compute_rectified_flow_fid_and_kid(model, real_loader, device, num_samples=1000, metric_batch_size=32):
    fid = FrechetInceptionDistance(
        feature=2048,
        normalize=True,
        reset_real_features=False,
    ).set_dtype(torch.float64).to(device)
    kid = KernelInceptionDistance(
        feature=2048,
        subsets=10,
        subset_size=100,
        normalize=True,
        reset_real_features=False,
    ).to(device)

    seen_real = 0
    for real_images, _ in tqdm(real_loader, desc="Rectified-flow real metrics", leave=False):
        remaining = num_samples - seen_real
        if remaining <= 0:
            break
        real_images = real_images[: min(metric_batch_size, remaining)].to(device)
        real_images = 0.5 * (real_images + 1.0)
        real_images = prepare_for_inception_metrics(real_images)
        fid.update(real_images, real=True)
        kid.update(real_images, real=True)
        seen_real += real_images.size(0)

    generated = 0
    pbar = tqdm(total=num_samples, desc="rectified-flow fake metrics", leave=False)
    while generated < num_samples:
        batch_n = min(metric_batch_size, num_samples - generated)
        fake_images = sample_rectified_flow(
            model,
            n_samples=batch_n,
            steps=image_ode_steps,
            show_progress=False,
        ).to(device)
        fake_images = prepare_for_inception_metrics(fake_images)
        fid.update(fake_images, real=False)
        kid.update(fake_images, real=False)
        generated += batch_n
        pbar.update(batch_n)
    pbar.close()

    kid_mean, kid_std = kid.compute()
    return {
        "fid": fid.compute().item(),
        "kid_mean": kid_mean.item(),
        "kid_std": kid_std.item(),
    }


rectified_flow_metrics = compute_rectified_flow_fid_and_kid(
    image_flow_model,
    fashion_loader,
    device,
    num_samples=1000,
    metric_batch_size=32,
)
print(rectified_flow_metrics)


These numbers should be read in the same spirit as the earlier model families. They matter because they let us compare distributions quantitatively, but they do not replace visual inspection, solver analysis, or compute-awareness. In continuous-time models, a mediocre score may reflect architecture limits, undertraining, weak path design, or a sampler that is simply too crude.


## Final Perspective

Flow matching is a transparent re-expression of a recurring theme in modern generative modeling: start from a simple source of randomness, choose a geometric structure that makes the generative problem manageable, and train a neural mechanism that turns that source into samples from the data distribution.

In VAEs the geometry lives in latent variables and variational inference. In GANs it lives in an adversarial discrepancy and a generator architecture. In diffusion it lives in a stochastic path plus a reverse field. In flow matching it becomes almost explicit: choose a path, define the local transport targets, learn the vector field, and integrate the resulting ODE. This makes the conceptual unity of modern generative modeling especially easy to see.
